# Bài 22: Nhận diện mã từ file upload
## User kéo file vào — hệ thống tự đoán mã, chỉ cần xác nhận

**Mục tiêu:**
- Viết `detect_symbol_from_pdf()` — đọc trang đầu PDF, LLM đoán ticker
- Bỏ ô 'Mã công ty cho tài liệu' — mỗi file tự nhận diện mã riêng
- Hiện mã đã đoán cho user **xác nhận/sửa** trước khi thêm (an toàn cho dữ liệu tài chính)

**Kết nối:** bài 21 rút mã từ *câu hỏi*. Bài này rút mã từ *tài liệu*. Cùng một ý tưởng — dùng LLM thay cho việc user gõ tay.

---
## Phần 1: Lý thuyết

### Vấn đề

Hiện tại một ô 'Mã công ty' áp cho **cả lô** file upload. Kéo cùng lúc `NVDA-10Q.pdf` và `TSLA-10Q.pdf` rồi gõ `NVDA` → file TSLA bị gắn nhãn sai. Muốn đúng phải upload từng công ty một đợt — tẻ nhạt.

---

### Giải pháp: đọc trang bìa → LLM đoán mã

Trang đầu báo cáo 10-Q luôn có tên công ty và thường có dòng *"Trading Symbol: NVDA"*. Ta đọc trang 1, đưa cho Gemini hỏi *"mã ticker của công ty này là gì?"* → trả `NVDA`.

```python
doc = fitz.open(stream=file_bytes, filetype="pdf")
first_page = doc[0].get_text()[:3000]   # 3000 ký tự đầu là đủ
# → đưa first_page cho LLM
```

Cách này khỏe hơn đoán theo tên file (vì `report.pdf` thì chịu), và chạy được cả báo cáo nội bộ.

---

### Vì sao PHẢI cho user xác nhận?

LLM đôi khi đoán sai. Gắn nhầm mã = truy xuất sai dữ liệu → nguy hiểm với số liệu tài chính. Nên thiết kế: tự đoán → **hiện ra ô sửa được** → user liếc qua, bấm thêm. User không phải *gõ*, chỉ *xác nhận*.

---

### Bẫy quan trọng: đừng đoán lại mỗi lần rerun

Streamlit **chạy lại toàn bộ script mỗi lần user tương tác** (gõ phím, bấm nút...). Nếu gọi `detect_symbol_from_pdf()` thẳng trong luồng → mỗi lần rerun lại gọi LLM cho mọi file → chậm và tốn.

→ Giải pháp: đoán **1 lần cho mỗi file**, lưu kết quả vào `st.session_state` theo key = tên file. Lần rerun sau thấy đã có → dùng lại, không gọi LLM nữa.

```python
key = f"detected::{file.name}"
if key not in st.session_state:                       # chỉ đoán khi CHƯA có
    st.session_state[key] = detect_symbol_from_pdf(file.getvalue())
```

---
## Phần 2: Ví dụ — đoán mã từ trang bìa

Đọc trang 1 của một PDF rồi cho Gemini đoán ticker.

In [1]:
import os
import fitz
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

DETECT_PROMPT = """Đây là trang đầu của một báo cáo tài chính.
Xác định MÃ TICKER cổ phiếu (sàn Mỹ) của công ty phát hành báo cáo.
- Trả về DUY NHẤT mã ticker in hoa, ví dụ: NVDA
- Không xác định được → trả về: UNKNOWN

Nội dung trang đầu:
{page_text}
"""

def detect_symbol_from_pdf(file_bytes: bytes) -> str:
    doc = fitz.open(stream=file_bytes, filetype="pdf")
    first_page = doc[0].get_text()[:3000] if doc.page_count else ""
    doc.close()
    prompt = DETECT_PROMPT.format(page_text=first_page)
    response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    return response.text.strip().upper()

# Thử với vài file trong phase4/data
for path in [
    "../phase4/data/NVDA-F1Q26-10-Q.pdf",
    "../phase4/data/AMD_F1Q26-10-Q.pdf",
]:
    with open(path, "rb") as f:
        print(f"{os.path.basename(path)} → {detect_symbol_from_pdf(f.read())}")

NVDA-F1Q26-10-Q.pdf → NVDA
AMD_F1Q26-10-Q.pdf → AMD


Trang bìa 10-Q có đủ thông tin để LLM đoán đúng mã. Giờ đưa vào app.

---
## Phần 3: Bài tập

### Bước 1: Thêm `detect_symbol_from_pdf()` vào `app/services/ingest.py`

Đầu file thêm import client (ingest.py chưa có):
```python
from config import client, MODEL_NAME
```

Rồi thêm hàm (điền `TODO` — có thêm sanity check + fallback so với ví dụ):

```python
DETECT_PROMPT = """Đây là trang đầu của một báo cáo tài chính.
Xác định MÃ TICKER cổ phiếu (sàn Mỹ) của công ty phát hành báo cáo.
- Trả về DUY NHẤT mã ticker in hoa, ví dụ: NVDA
- Không xác định được → trả về: UNKNOWN

Nội dung trang đầu:
{page_text}
"""

def detect_symbol_from_pdf(file_bytes: bytes) -> str:
    """Đọc trang đầu PDF, dùng LLM đoán mã ticker. Trả 'UNKNOWN' nếu không chắc."""
    doc = fitz.open(stream=file_bytes, filetype="pdf")
    first_page = doc[0].get_text()[:3000] if doc.page_count else ""
    doc.close()

    if not first_page.strip():
        return "UNKNOWN"

    try:
        prompt = DETECT_PROMPT.format(page_text=first_page)
        response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
        symbol = response.text.strip().upper()
        # TODO: sanity check — ticker hợp lệ thường ngắn & chỉ chữ/số.
        #   Nếu symbol rỗng, hoặc dài hơn 6 ký tự, hoặc không phải chữ-số
        #   → coi như không chắc, return "UNKNOWN"
        if not symbol or len(symbol) > 6 or not symbol.isalnum():
            return "UNKNOWN"
        return symbol
    except Exception as e:
        logger.error(f"[detect] lỗi: {e}")
        return "UNKNOWN"
```

---

### Bước 2: Sửa phần upload trong `app/main.py`

Thay **toàn bộ** khối upload cũ (ô 'Mã công ty' + file_uploader + nút Thêm) bằng luồng mới: mỗi file tự đoán mã (cache trong session_state) + ô sửa được.

Thêm import: `from services.ingest import ingest_pdf, detect_symbol_from_pdf`

```python
    st.markdown("---")
    st.subheader("📎 Tải lên báo cáo")

    uploaded_files = st.file_uploader(
        "Chọn PDF", type="pdf", accept_multiple_files=True
    )

    confirmed = {}   # {tên_file: mã user xác nhận}
    if uploaded_files:
        st.caption("Mã nhận diện tự động — sửa nếu sai:")
        for file in uploaded_files:
            # TODO: đoán 1 lần cho mỗi file, lưu vào session_state
            #   key = f"detected::{file.name}"
            #   if key not in st.session_state:
            #       with st.spinner(f"Đang nhận diện {file.name}..."):
            #           st.session_state[key] = detect_symbol_from_pdf(file.getvalue())

            # TODO: ô nhập mã cho file này, value = mã đã đoán, key riêng theo tên file
            #   confirmed[file.name] = st.text_input(
            #       f"Mã cho `{file.name}`",
            #       value=st.session_state[key],
            #       key=f"confirm::{file.name}",
            #   )
            pass

    if st.button("➕ Thêm vào hệ thống"):
        if not uploaded_files:
            st.warning("Vui lòng chọn ít nhất một file PDF.")
        else:
            for file in uploaded_files:
                symbol = confirmed[file.name].upper().strip()
                # TODO: nếu symbol rỗng hoặc == "UNKNOWN" → st.warning + bỏ qua (continue)
                # Nếu hợp lệ → ingest_pdf(file.getvalue(), symbol, file.name)
                #   rồi st.success(f"Đã thêm {n} chunks từ {file.name} ({symbol})")
                pass
```

> Ghi chú: ô nhập cần `key` riêng theo tên file (`key=f"confirm::{file.name}"`) để Streamlit phân biệt các ô — nếu không, nhiều file sẽ dùng chung 1 ô và lỗi 'duplicate key'.

---

### Bước 3: Test

`streamlit run app/main.py`, mở phần Tải lên:
1. Kéo **cùng lúc** 2 file khác công ty (vd NVDA 10-Q và AMD 10-Q trong `phase4/data/`)
2. Mỗi file hiện 1 ô mã, **tự điền** đúng `NVDA` / `AMD` — không phải gõ
3. Bấm **Thêm vào hệ thống** → mỗi file ingest đúng mã của nó
4. Mục 📚 Kho tài liệu hiện cả NVDA và AMD

**Checklist:**
- [ ] Không còn ô 'Mã công ty' chung
- [ ] Kéo nhiều công ty 1 lần → mỗi file nhận đúng mã riêng
- [ ] Sửa tay mã đoán sai được, rồi thêm vẫn đúng
- [ ] Gõ phím ở ô mã KHÔNG làm gọi lại LLM (nhờ cache session_state) — nhìn log không thấy `[detect]` lặp